# Tutorial: Prediction History Tracking in NFL ML API

Audience:
- You are building/maintaining the NFL prediction backend.

Prerequisites:
- Basic Python, JSON, and FastAPI familiarity.

Learning goals:
- Persist prediction events safely.
- Keep history bounded and queryable.
- Integrate history writes directly into prediction flow.


## Outline

1. Setup paths and utilities
2. Define a canonical history record
3. Build a JSON history store with locking
4. Add append/read/query workflows
5. Connect to FastAPI `/api/predict` and `/api/history`
6. Exercise


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from threading import Lock
from typing import Any
import json

REPO_ROOT = Path.cwd().resolve()
REAL_HISTORY_PATH = REPO_ROOT / 'backend' / 'Predictions' / 'prediction_history.json'
DEMO_HISTORY_PATH = REPO_ROOT / 'output' / 'jupyter-notebook' / 'demo_prediction_history.json'

REPO_ROOT, REAL_HISTORY_PATH


## Step 1 - Define a canonical history record

A stable schema prevents frontend drift and keeps analytics simple.


In [ ]:
@dataclass
class PredictionHistoryRecord:
    ts: str
    game_id: str
    season: int
    week: int
    home_team: str
    away_team: str
    home_score: float
    away_score: float
    point_diff: float
    home_win_probability: float
    away_win_probability: float
    prediction_source: str

    @classmethod
    def from_prediction(cls, prediction: dict[str, Any]) -> 'PredictionHistoryRecord':
        now = datetime.now(timezone.utc).isoformat()
        return cls(
            ts=prediction.get('ts') or now,
            game_id=str(prediction.get('game_id') or '').strip(),
            season=int(prediction.get('season')),
            week=int(prediction.get('week')),
            home_team=str(prediction.get('home_team')).upper(),
            away_team=str(prediction.get('away_team')).upper(),
            home_score=float(prediction.get('home_score')),
            away_score=float(prediction.get('away_score')),
            point_diff=float(prediction.get('point_diff')),
            home_win_probability=float(prediction.get('home_win_probability')),
            away_win_probability=float(prediction.get('away_win_probability')),
            prediction_source=str(prediction.get('prediction_source') or 'model'),
        )

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


## Step 2 - Build a small JSON history store

Use a lock + max-entry cap so concurrent writes stay safe and file size stays bounded.


In [ ]:
class JsonPredictionHistoryStore:
    def __init__(self, path: Path, max_entries: int = 1000):
        self.path = path
        self.max_entries = max_entries
        self._lock = Lock()
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def load(self) -> list[dict[str, Any]]:
        if not self.path.exists():
            return []
        try:
            raw = json.loads(self.path.read_text(encoding='utf-8'))
            return raw if isinstance(raw, list) else []
        except Exception:
            return []

    def save(self, records: list[dict[str, Any]]) -> None:
        self.path.write_text(json.dumps(records, indent=2), encoding='utf-8')

    def append(self, record: dict[str, Any], dedupe_by_game: bool = True) -> None:
        with self._lock:
            records = self.load()
            if dedupe_by_game and record.get('game_id'):
                gid = record['game_id']
                records = [r for r in records if r.get('game_id') != gid]
            records.insert(0, record)
            records = records[: self.max_entries]
            self.save(records)

    def latest(self, limit: int = 20) -> list[dict[str, Any]]:
        return self.load()[:limit]

    def by_team(self, team: str, limit: int = 20) -> list[dict[str, Any]]:
        t = str(team).upper()
        rows = [r for r in self.load() if r.get('home_team') == t or r.get('away_team') == t]
        return rows[:limit]


## Step 3 - Write and query demo prediction history


In [ ]:
store = JsonPredictionHistoryStore(DEMO_HISTORY_PATH, max_entries=5)

sample_predictions = [
    {
        'game_id': '2026-1-KC-BUF', 'season': 2026, 'week': 1,
        'home_team': 'KC', 'away_team': 'BUF',
        'home_score': 27.4, 'away_score': 23.9, 'point_diff': 3.5,
        'home_win_probability': 0.61, 'away_win_probability': 0.39,
        'prediction_source': 'schedule_row'
    },
    {
        'game_id': '2026-1-SF-LAR', 'season': 2026, 'week': 1,
        'home_team': 'SF', 'away_team': 'LAR',
        'home_score': 24.0, 'away_score': 20.2, 'point_diff': 3.8,
        'home_win_probability': 0.58, 'away_win_probability': 0.42,
        'prediction_source': 'roll_forward'
    },
]

for pred in sample_predictions:
    record = PredictionHistoryRecord.from_prediction(pred).to_dict()
    store.append(record)

store.latest()


## Step 4 - Team filter and bounded history behavior


In [ ]:
# Add duplicate game_id to prove dedupe behavior.
updated = sample_predictions[0] | {'home_score': 28.2, 'point_diff': 4.3}
store.append(PredictionHistoryRecord.from_prediction(updated).to_dict(), dedupe_by_game=True)

print('Latest count:', len(store.latest()))
print('KC rows:', len(store.by_team('KC')))
store.latest(3)


## Step 5 - FastAPI integration pattern

Drop this pattern into your backend: append after each successful prediction and expose `/api/history`.


In [ ]:
integration_snippet = '''
# inside backend/main.py
history_store = JsonPredictionHistoryStore(Path('backend/Predictions/prediction_history.json'))

@app.post('/api/predict')
async def predict(req: PredictionRequest):
    payload = run_model(req)
    history_store.append(payload, dedupe_by_game=True)
    return payload

@app.get('/api/history')
async def history(limit: int = 100):
    rows = history_store.latest(limit=limit)
    return {'entries': rows, 'total': len(rows)}
'''

print(integration_snippet)


## Step 6 - Read your real project history file (if present)


In [ ]:
if REAL_HISTORY_PATH.exists():
    real_rows = json.loads(REAL_HISTORY_PATH.read_text(encoding='utf-8'))
    print('Real history entries:', len(real_rows))
    print('Most recent entry keys:', sorted(real_rows[0].keys()) if real_rows else [])
else:
    print('No real history file found yet at', REAL_HISTORY_PATH)


## Exercises

- Add a `model_version` field to each record and persist it.
- Add `by_week(season, week)` query helper.
- Add a daily backup file (for example `prediction_history_YYYYMMDD.json`).


In [ ]:
# Exercise scaffold
def by_week(records: list[dict[str, Any]], season: int, week: int) -> list[dict[str, Any]]:
    return [r for r in records if int(r.get('season', -1)) == season and int(r.get('week', -1)) == week]

by_week(store.latest(50), season=2026, week=1)
